[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/39_ml_systems/03_packaging_and_model_registry.ipynb)

# Bài 03 — Đóng gói mô hình và Model Registry

**🎯 Mục tiêu**
- Đóng gói **toàn bộ** đường đi từ dữ liệu thô → dự đoán thành một đối tượng; lưu, nạp lại trong **tiến trình khác** và kiểm chứng.
- Hiểu vì sao `pickle`/`joblib` **không an toàn** với file không tin cậy; dùng `skops` để nạp có kiểm soát.
- Gói kèm **metadata**: phiên bản thư viện, schema đặc trưng, metric, hash dữ liệu, hash file mô hình (toàn vẹn); sinh **model card**.
- **Kiểm tra đầu vào** bằng schema (pydantic) — mô hình không tự báo lỗi khi nhận dữ liệu sai.
- Tự viết **model registry** (phiên bản, alias champion, cổng thăng hạng, rollback); dùng MLflow Model Registry; xuất mô hình sang **ONNX** từ đầu.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/39_ml_systems

In [ ]:
import hashlib
import inspect
import json
import os
import pickle
import pickletools
import subprocess
import sys
import tempfile
import textwrap
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path
from typing import Literal

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

from ml_course.datasets import make_customer_churn

try:
    from pydantic import BaseModel, ConfigDict, Field, ValidationError
except ImportError as e:
    raise ImportError('Bài này cần pydantic: pip install -e ".[mlops]"') from e

WORKDIR = Path(tempfile.mkdtemp())                                      # mọi file của bài được ghi vào thư mục tạm

## 1. Gói mô hình = pipeline đầy đủ

Thứ được triển khai **không phải** "mô hình" mà là hàm `dữ liệu thô → dự đoán`. Mọi bước tiền xử lý (điền thiếu, chuẩn hoá, one-hot) phải nằm **trong** đối tượng được lưu — nếu không, đội phục vụ sẽ phải viết lại chúng và gây lệch train/serving (bài 01).

In [ ]:
data = make_customer_churn(6000, seed=0)
FEATURES = ["tenure_months", "contract", "internet_service", "monthly_charges", "total_charges",
            "payment_method", "support_calls", "senior", "n_products"]
NUMERIC = ["tenure_months", "monthly_charges", "total_charges", "support_calls", "n_products"]
CATEGORICAL = ["contract", "internet_service", "payment_method", "senior"]
X_tr, X_te, y_tr, y_te = train_test_split(data[FEATURES], data.churn.values, test_size=0.3, random_state=0, stratify=data.churn)


def make_preprocess():
    return ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUMERIC),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
    ])


model = make_pipeline(make_preprocess(), LogisticRegression(max_iter=1000)).fit(X_tr, y_tr)
print(f"ROC-AUC test = {roc_auc_score(y_te, model.predict_proba(X_te)[:, 1]):.4f}")
print("đầu vào là DataFrame thô:", X_te.iloc[:1].to_dict("records")[0])

## 2. Lưu, nạp lại ở tiến trình khác — và cái bẫy `__main__`

Phép thử đúng cho một gói mô hình: một **tiến trình Python mới** (như server phục vụ) nạp file và cho **đúng** dự đoán như notebook.

In [ ]:
model_path = WORKDIR / "churn_model.joblib"
joblib.dump(model, model_path)
sample_path = WORKDIR / "sample.csv"
X_te.iloc[:5].to_csv(sample_path, index=False)
loader = WORKDIR / "load_and_predict.py"
loader.write_text(textwrap.dedent("""
    import json, sys
    import joblib, pandas as pd
    model = joblib.load(sys.argv[1])
    print(json.dumps(model.predict_proba(pd.read_csv(sys.argv[2]))[:, 1].tolist()))
"""), encoding="utf-8")


def predict_in_fresh_process(path):
    result = subprocess.run([sys.executable, str(loader), str(path), str(sample_path)], capture_output=True, text=True)
    if result.returncode != 0:
        return "LỖI: " + result.stderr.strip().splitlines()[-1]
    return np.array(json.loads(result.stdout))


fresh = predict_in_fresh_process(model_path)
print("tiến trình mới :", np.round(fresh, 6))
print("notebook       :", np.round(model.predict_proba(X_te.iloc[:5])[:, 1], 6))
print("khớp:", np.allclose(fresh, model.predict_proba(X_te.iloc[:5])[:, 1]), f"| kích thước file: {model_path.stat().st_size / 1024:.1f} KB")

Pickle **không lưu code** — nó lưu *tham chiếu* tới hàm/lớp (`module.tên`) và dữ liệu. Nếu pipeline dùng một hàm định nghĩa **trong notebook** (module `__main__`), tiến trình khác không tìm thấy hàm đó:

In [ ]:
def add_charge_ratio(df):
    df = df.copy()
    df["charge_per_product"] = df.monthly_charges / df.n_products
    return df


custom = make_pipeline(FunctionTransformer(add_charge_ratio),
                       ColumnTransformer([("num", StandardScaler(), NUMERIC + ["charge_per_product"]),
                                          ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)]),
                       LogisticRegression(max_iter=1000)).fit(X_tr, y_tr)
custom_path = WORKDIR / "custom_model.joblib"
joblib.dump(custom, custom_path)
print(predict_in_fresh_process(custom_path))

Cách khắc phục: đặt code tuỳ biến (transformer, hàm đặc trưng) trong một **module/package có thể import** (ví dụ `src/churn/features.py`), cài package đó ở cả môi trường train và serve, và ghi **phiên bản package** vào metadata. Code và mô hình phải đi cùng nhau.

## 3. Pickle không an toàn

Khi nạp, pickle có thể **gọi bất kỳ hàm nào** — `__reduce__` cho phép đối tượng chỉ định "hãy gọi hàm X với tham số Y để dựng lại tôi". Ví dụ vô hại dưới đây chỉ gọi `print`; kẻ tấn công sẽ gọi `os.system`.

In [ ]:
class Payload:
    def __reduce__(self):
        return (print, ("⚠️  code này chạy ngay khi pickle.loads — không cần gọi hàm nào của mô hình!",))


malicious = pickle.dumps(Payload())
_ = pickle.loads(malicious)
print("\nopcode của file — thấy rõ hàm sẽ bị gọi:")
pickletools.dis(malicious)

Quy tắc: **không bao giờ** `pickle.load`/`joblib.load`/`torch.load(weights_only=False)` file từ nguồn không tin cậy (tải trên mạng, người dùng upload). Các lựa chọn an toàn hơn:
- **skops** (`skops.io`) cho scikit-learn: định dạng không thực thi code; khi nạp liệt kê các kiểu **chưa được tin cậy** để ta duyệt.
- **safetensors** cho trọng số mạng nơ-ron (chỉ tensor, không code); `torch.load(..., weights_only=True)`.
- **ONNX** (phần 8): đồ thị tính toán thuần, không chứa code Python.
- Ký số/hash file mô hình và chỉ nạp file có hash khớp với registry (phần 4).

In [ ]:
try:
    import skops.io as sio
except ImportError:
    sio = None
    print('Chưa cài skops — bỏ qua: pip install -e ".[mlops]"')

if sio is not None:
    blob = sio.dumps(model)
    untrusted = sio.get_untrusted_types(data=blob)
    print("kiểu cần duyệt trước khi nạp:", untrusted)
    restored = sio.loads(blob, trusted=untrusted)                       # chỉ tin đúng các kiểu đã duyệt
    print("khớp:", np.allclose(restored.predict_proba(X_te)[:, 1], model.predict_proba(X_te)[:, 1]))
    suspicious = sio.dumps(Payload())
    print("file đáng ngờ chứa:", sio.get_untrusted_types(data=suspicious))
    try:
        sio.loads(suspicious)                                           # không truyền trusted → từ chối
    except Exception as err:  # noqa: BLE001
        print("skops từ chối nạp:", type(err).__name__)

## 4. Metadata, schema và tính toàn vẹn

Một file mô hình "trần" không trả lời được: train trên dữ liệu nào? metric bao nhiêu? cần phiên bản thư viện nào? đầu vào phải có dạng gì? Ta lưu một **gói**: `model.joblib` + `metadata.json`.

In [ ]:
def infer_schema(df: pd.DataFrame) -> dict:
    """Schema đầu vào học từ dữ liệu train: kiểu, khoảng giá trị, tập giá trị hợp lệ."""
    schema = {}
    for col in df.columns:
        s = df[col]
        if s.dtype == bool or s.dtype == object:
            schema[col] = {"kind": "category", "values": sorted(map(str, s.unique()))}
        else:
            schema[col] = {"kind": "number", "min": float(s.min()), "max": float(s.max())}
    return schema


def sha256_file(path: Path) -> str:
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def save_package(model, directory: Path, metadata: dict) -> Path:
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    joblib.dump(model, directory / "model.joblib")
    full = {**metadata, "model_sha256": sha256_file(directory / "model.joblib"),
            "versions": {"python": sys.version.split()[0], "sklearn": sklearn.__version__, "numpy": np.__version__, "pandas": pd.__version__},
            "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")}
    (directory / "metadata.json").write_text(json.dumps(full, indent=2, ensure_ascii=False), encoding="utf-8")
    return directory


def load_package(directory: Path):
    directory = Path(directory)
    meta = json.loads((directory / "metadata.json").read_text(encoding="utf-8"))
    if sha256_file(directory / "model.joblib") != meta["model_sha256"]:
        raise RuntimeError("model.joblib không khớp hash trong metadata — file bị sửa hoặc hỏng, từ chối nạp")
    if meta["versions"]["sklearn"] != sklearn.__version__:
        warnings.warn(f"mô hình lưu bằng scikit-learn {meta['versions']['sklearn']}, đang chạy {sklearn.__version__} — có thể sai lệch", stacklevel=2)
    return joblib.load(directory / "model.joblib"), meta


test_auc = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])
data_hash = hashlib.sha256(pd.util.hash_pandas_object(data, index=False).values.tobytes()).hexdigest()[:16]
package_dir = save_package(model, WORKDIR / "churn_lr", {
    "name": "churn", "description": "Dự đoán xác suất khách hàng rời bỏ trong kỳ tới",
    "features": FEATURES, "schema": infer_schema(X_tr), "target": "churn (1 = rời bỏ)",
    "metrics": {"test_roc_auc": round(test_auc, 4)}, "training_data": {"source": "make_customer_churn(6000, seed=0)", "hash": data_hash, "n_train": len(X_tr)},
})
meta = json.loads((package_dir / "metadata.json").read_text(encoding="utf-8"))
print(json.dumps({k: meta[k] for k in ["name", "metrics", "training_data", "versions"]}, indent=2, ensure_ascii=False))
print("schema (2 cột):", {k: meta["schema"][k] for k in ["contract", "monthly_charges"]})

In [ ]:
loaded, _ = load_package(package_dir)
print("nạp gói hợp lệ: OK")
with open(package_dir / "model.joblib", "ab") as f:                    # mô phỏng file bị thay đổi
    f.write(b"\x00")
try:
    load_package(package_dir)
except RuntimeError as err:
    print("file bị sửa:", err)
_ = save_package(model, package_dir, {k: meta[k] for k in meta if k not in ("model_sha256", "versions", "created_utc")})   # ghi lại gói sạch

## 5. Kiểm tra đầu vào: mô hình không tự báo lỗi

Thử đưa vào những bản ghi **sai** — mô hình vẫn vui vẻ trả về một xác suất:

In [ ]:
good = X_te.iloc[0].to_dict()
bad_inputs = {
    "hợp lệ": good,
    "hợp đồng gõ sai ('monthly')": {**good, "contract": "monthly"},
    "cước âm": {**good, "monthly_charges": -50.0},
    "cước theo nghìn đồng (×1000)": {**good, "monthly_charges": good["monthly_charges"] * 1000},
}
for name, record in bad_inputs.items():
    print(f"{name:30s} → P(rời bỏ) = {model.predict_proba(pd.DataFrame([record]))[0, 1]:.3f}")

`OneHotEncoder(handle_unknown="ignore")` biến giá trị lạ thành vector toàn 0, chuẩn hoá biến số âm thành một z-score bình thường — **không có lỗi nào**, chỉ có dự đoán sai một cách tự tin. Hệ thống phải từ chối đầu vào sai **trước** khi tới mô hình. Ở API (chương 40) dùng **pydantic**:

In [ ]:
class CustomerFeatures(BaseModel):
    model_config = ConfigDict(extra="forbid")                           # trường lạ (gõ sai tên) → lỗi
    tenure_months: int = Field(ge=0, le=600)
    contract: Literal["month-to-month", "one_year", "two_year"]
    internet_service: Literal["dsl", "fiber", "none"]
    monthly_charges: float = Field(gt=0, le=1000)
    total_charges: float = Field(ge=0)
    payment_method: Literal["bank_transfer", "cash", "credit_card", "e-wallet"]
    support_calls: int = Field(ge=0, le=100)
    senior: bool
    n_products: int = Field(ge=1, le=20)


tests = {**bad_inputs, "gõ sai tên trường": {**{k: v for k, v in good.items() if k != "tenure_months"}, "tenure_month": 5}}
for name, record in tests.items():
    try:
        CustomerFeatures(**record)
        print(f"{name:30s} → hợp lệ")
    except ValidationError as err:
        problems = "; ".join(f"{'.'.join(map(str, e['loc']))}: {e['msg']}" for e in err.errors())
        print(f"{name:30s} → TỪ CHỐI ({problems})")

Cước ×1000 vẫn lọt qua vì nằm trong giới hạn nghiệp vụ (≤ 1000). Lớp kiểm tra thứ hai so với **schema học từ dữ liệu train** (lưu trong metadata): giá trị ngoài khoảng đã thấy không phải lúc nào cũng sai, nhưng mô hình chưa từng học vùng đó → cảnh báo/log để giám sát (chương 40).

In [ ]:
def check_against_schema(df: pd.DataFrame, schema: dict, tolerance: float = 0.1) -> list[str]:
    """Trả về danh sách vấn đề: thiếu cột, giá trị thiếu, danh mục lạ, số ngoài khoảng train (nới `tolerance` × biên độ)."""
    issues = [f"thiếu cột {c}" for c in schema if c not in df.columns]
    for col, spec in schema.items():
        if col not in df.columns:
            continue
        s = df[col]
        if s.isna().any():
            issues.append(f"{col}: {int(s.isna().sum())} giá trị thiếu")
        if spec["kind"] == "category":
            unknown = set(map(str, s.dropna().unique())) - set(spec["values"])
            if unknown:
                issues.append(f"{col}: giá trị lạ {sorted(unknown)}")
        else:
            span = spec["max"] - spec["min"]
            lo, hi = spec["min"] - tolerance * span, spec["max"] + tolerance * span
            n_out = int(((s < lo) | (s > hi)).sum())
            if n_out:
                issues.append(f"{col}: {n_out} giá trị ngoài [{lo:.1f}, {hi:.1f}]")
    return issues


batch = pd.DataFrame([bad_inputs["hợp lệ"], bad_inputs["cước theo nghìn đồng (×1000)"], bad_inputs["hợp đồng gõ sai ('monthly')"]])
print("lô dữ liệu test:", check_against_schema(X_te, meta["schema"]) or "không có vấn đề")
print("lô có lỗi      :", check_against_schema(batch, meta["schema"]))

## 6. Model card

Model card (Mitchell et al., 2019) là "nhãn dinh dưỡng" của mô hình: dùng để làm gì, **không** nên dùng cho việc gì, hiệu năng trên từng nhóm, dữ liệu, hạn chế. Sinh tự động từ metadata để luôn đồng bộ với mô hình:

In [ ]:
def segment_auc(model, X, y, column):
    out = {}
    for value in sorted(X[column].unique()):
        mask = (X[column] == value).values
        if len(np.unique(y[mask])) == 2:
            out[str(value)] = roc_auc_score(y[mask], model.predict_proba(X[mask])[:, 1])
    return out


def model_card(meta: dict, segments: dict) -> str:
    seg_rows = "\n".join(f"| {k} | {v:.3f} |" for k, v in segments.items())
    return f"""# Model card — {meta['name']}

**Mô tả:** {meta['description']}
**Ngày tạo:** {meta['created_utc']} · scikit-learn {meta['versions']['sklearn']} · sha256 `{meta['model_sha256'][:12]}…`

## Mục đích sử dụng
- Dùng để: xếp hạng khách hàng cho chiến dịch giữ chân (con người ra quyết định cuối cùng).
- Không dùng để: từ chối dịch vụ, định giá cá nhân, hoặc bất kỳ quyết định tự động nào ảnh hưởng quyền lợi khách hàng.

## Dữ liệu
Nguồn: {meta['training_data']['source']} · hash `{meta['training_data']['hash']}` · {meta['training_data']['n_train']:,} dòng train. Đầu vào: {len(meta['features'])} đặc trưng (schema trong metadata).

## Hiệu năng
ROC-AUC test = {meta['metrics']['test_roc_auc']} · theo loại hợp đồng:

| contract | ROC-AUC |
|---|---|
{seg_rows}

## Hạn chế
- Dữ liệu mô phỏng; chưa kiểm định trên khách hàng thật. Hiệu năng giảm nếu phân phối đầu vào trôi (chương 40).
- Xác suất chưa được hiệu chỉnh riêng (chương 37) — dùng để xếp hạng, không đọc như xác suất tuyệt đối.
"""


meta = json.loads((package_dir / "metadata.json").read_text(encoding="utf-8"))
print(model_card(meta, segment_auc(model, X_te, y_te, "contract")))

AUC **trong từng nhóm** hợp đồng đều thấp hơn AUC tổng thể (nhóm month-to-month thấp hơn hẳn): một phần khả năng xếp hạng của mô hình đến từ việc phân biệt **giữa** các nhóm (khách month-to-month rời bỏ nhiều hơn hẳn). Model card buộc ta nói rõ điều này — nếu chiến dịch chỉ nhắm khách month-to-month, AUC hữu ích là con số của nhóm đó, không phải con số tổng.

## 7. Model registry

Registry là "kho mô hình có phiên bản": mỗi lần đăng ký tạo **phiên bản** bất biến; các **alias** (`champion`, `challenger`) trỏ tới phiên bản; hệ thống phục vụ nạp `churn@champion` — đổi mô hình = đổi con trỏ, **rollback** = trỏ lại phiên bản cũ, không cần huấn luyện lại hay deploy lại code.

In [ ]:
class ModelRegistry:
    """Registry tối giản trên hệ thống file: <root>/<name>/v<k>/{model.joblib, metadata.json} + aliases.json."""

    def __init__(self, root):
        self.root = Path(root)

    def _aliases_path(self, name):
        return self.root / name / "aliases.json"

    def aliases(self, name) -> dict:
        p = self._aliases_path(name)
        return json.loads(p.read_text(encoding="utf-8")) if p.exists() else {}

    def register(self, name, model, metadata) -> int:
        existing = [int(p.name[1:]) for p in (self.root / name).glob("v*")] if (self.root / name).exists() else []
        version = max(existing, default=0) + 1
        save_package(model, self.root / name / f"v{version}", {**metadata, "name": name, "version": version})
        return version

    def set_alias(self, name, alias, version):
        if not (self.root / name / f"v{version}").exists():
            raise KeyError(f"{name} không có phiên bản {version}")
        aliases = self.aliases(name)
        history = aliases.get("_history", [])
        history.append({"alias": alias, "from": aliases.get(alias), "to": version, "at": datetime.now(timezone.utc).isoformat(timespec="seconds")})
        aliases.update({alias: version, "_history": history})
        self._aliases_path(name).write_text(json.dumps(aliases, indent=2), encoding="utf-8")

    def load(self, uri: str):
        """'name@alias' hoặc 'name/version'."""
        if "@" in uri:
            name, alias = uri.split("@")
            version = self.aliases(name)[alias]
        else:
            name, v = uri.split("/")
            version = int(v)
        return load_package(self.root / name / f"v{version}")

    def versions(self, name) -> pd.DataFrame:
        aliases = {v: a for a, v in self.aliases(name).items() if a != "_history"}
        rows = []
        for p in sorted((self.root / name).glob("v*"), key=lambda p: int(p.name[1:])):
            m = json.loads((p / "metadata.json").read_text(encoding="utf-8"))
            rows.append({"version": m["version"], "model": m["model_type"], **m["metrics"], "alias": aliases.get(m["version"], "")})
        return pd.DataFrame(rows)

**Cổng thăng hạng** (promotion gate): một phiên bản mới chỉ thành champion nếu qua mọi kiểm tra trên một tập **holdout cố định** — không chỉ "AUC tổng cao hơn".

In [ ]:
X_hold, y_hold = X_te, y_te                                            # holdout cố định cho mọi lần so sánh


def p99_latency_ms(model, row, n=200):
    times = []
    for _ in range(n):
        t0 = time.perf_counter()
        model.predict_proba(row)
        times.append((time.perf_counter() - t0) * 1000)
    return float(np.percentile(times, 99))


def promotion_gate(candidate, champion, min_gain=0.0, max_segment_drop=0.02, latency_budget_ms=20.0):
    cand_auc = roc_auc_score(y_hold, candidate.predict_proba(X_hold)[:, 1])
    champ_auc = roc_auc_score(y_hold, champion.predict_proba(X_hold)[:, 1])
    cand_seg, champ_seg = segment_auc(candidate, X_hold, y_hold, "contract"), segment_auc(champion, X_hold, y_hold, "contract")
    worst_drop = max(champ_seg[k] - cand_seg[k] for k in champ_seg)
    latency = p99_latency_ms(candidate, X_hold.iloc[[0]])
    checks = {
        f"AUC tổng {cand_auc:.4f} ≥ champion {champ_auc:.4f} + {min_gain}": cand_auc >= champ_auc + min_gain,
        f"không nhóm nào giảm > {max_segment_drop} (giảm nhiều nhất {worst_drop:+.4f})": worst_drop <= max_segment_drop,
        f"độ trễ p99 {latency:.2f} ms ≤ {latency_budget_ms} ms": latency <= latency_budget_ms,
        "schema đầu vào không đổi": list(getattr(candidate, "feature_names_in_", [])) == list(getattr(champion, "feature_names_in_", [])),
    }
    return all(checks.values()), checks


def train_model(kind, X, y):
    estimator = {"hgb": HistGradientBoostingClassifier(random_state=0), "logreg": LogisticRegression(max_iter=1000)}[kind]
    m = make_pipeline(make_preprocess(), estimator).fit(X, y)
    return m, {"holdout_roc_auc": round(roc_auc_score(y_hold, m.predict_proba(X_hold)[:, 1]), 4)}


registry = ModelRegistry(WORKDIR / "registry")
schema = infer_schema(X_tr)
m1, metrics1 = train_model("hgb", X_tr, y_tr)
v1 = registry.register("churn", m1, {"model_type": "HGB", "metrics": metrics1, "schema": schema})
registry.set_alias("churn", "champion", v1)
print(registry.versions("churn").to_string(index=False))

Phiên bản 2: hồi quy logistic. Phiên bản 3: cùng mô hình nhưng huấn luyện trên dữ liệu bị **lỗi pipeline dữ liệu** — cột `support_calls` bị ghi toàn 0 (một join hỏng).

In [ ]:
m2, metrics2 = train_model("logreg", X_tr, y_tr)
v2 = registry.register("churn", m2, {"model_type": "LogReg", "metrics": metrics2, "schema": schema})
X_broken = X_tr.assign(support_calls=0)
m3, metrics3 = train_model("logreg", X_broken, y_tr)
v3 = registry.register("churn", m3, {"model_type": "LogReg (dữ liệu lỗi)", "metrics": metrics3, "schema": infer_schema(X_broken)})

for version in [v2, v3]:
    champion, champ_meta = registry.load("churn@champion")
    candidate, _ = registry.load(f"churn/{version}")
    passed, checks = promotion_gate(candidate, champion)
    print(f"\nỨng viên v{version} vs champion v{champ_meta['version']}:")
    for desc, ok in checks.items():
        print(f"  {'✅' if ok else '❌'} {desc}")
    if passed:
        registry.set_alias("churn", "champion", version)
        print(f"  → thăng hạng v{version} thành champion")
    else:
        print("  → giữ nguyên champion")
print()
print(registry.versions("churn").to_string(index=False))
print("schema v3 · support_calls:", registry.load(f"churn/{v3}")[1]["schema"]["support_calls"])

Phiên bản 3 bị chặn: AUC giảm cả tổng thể lẫn trong từng nhóm. Lưu ý: chính schema học từ dữ liệu lỗi cũng là tín hiệu — `support_calls` có min = max = 0 (cột hằng). Một kiểm tra dữ liệu trước khi train ("cột không được hằng số", "phân phối không đổi quá nhiều so với lần trước") sẽ bắt lỗi còn sớm hơn.

**Rollback**: giả sử champion mới gây sự cố trên production — chỉ cần trỏ alias về phiên bản trước; lịch sử thay đổi alias được ghi lại để kiểm toán.

In [ ]:
registry.set_alias("churn", "champion", v1)                             # rollback tức thì
_, meta_now = registry.load("churn@champion")
print("champion hiện tại: v", meta_now["version"], sep="")
for event in registry.aliases("churn")["_history"]:
    print(f"  {event['alias']}: {'(chưa có)' if event['from'] is None else 'v' + str(event['from'])} → v{event['to']}")

## 8. MLflow Model Registry

Cùng khái niệm — phiên bản, alias, URI `models:/tên@alias` — trên MLflow (backend SQLite trong thư mục tạm).

In [ ]:
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
os.environ.setdefault("MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR", "false")
warnings.filterwarnings("ignore", message="IProgress not found")
try:
    import mlflow
    import mlflow.sklearn
    from mlflow.models import infer_signature
except ImportError:
    mlflow = None
    print('Chưa cài MLflow — bỏ qua: pip install -e ".[mlops]"')

if mlflow is not None:
    import logging
    logging.getLogger("mlflow").setLevel(logging.ERROR)
    mlflow.set_tracking_uri("sqlite:///" + (WORKDIR / "mlflow.db").as_posix())
    warnings.filterwarnings("ignore", message="Hint: Inferred schema contains integer")   # gợi ý về cột số nguyên có giá trị thiếu
    mlflow.set_experiment("churn-registry")
    for kind, m in [("hgb", m1), ("logreg", m2)]:
        with mlflow.start_run(run_name=kind):
            mlflow.log_metric("holdout_roc_auc", roc_auc_score(y_hold, m.predict_proba(X_hold)[:, 1]))
            signature = infer_signature(X_hold.iloc[:5], m.predict_proba(X_hold.iloc[:5])[:, 1])
            extra = {}
            if sio is not None and "skops_trusted_types" in inspect.signature(mlflow.sklearn.log_model).parameters:
                # MLflow 3 mặc định lưu mô hình sklearn bằng skops (phần 3) → khai báo các kiểu đã duyệt
                extra["skops_trusted_types"] = sio.get_untrusted_types(data=sio.dumps(m))
            mlflow.sklearn.log_model(m, name="model", registered_model_name="churn", signature=signature, **extra)
    client = mlflow.MlflowClient()
    print([(v.version, v.run_id[:8]) for v in client.search_model_versions("name='churn'")])
    client.set_registered_model_alias("churn", "champion", "2")
    served = mlflow.sklearn.load_model("models:/churn@champion")
    print("models:/churn@champion →", type(served[-1]).__name__,
          "| khớp:", np.allclose(served.predict_proba(X_hold)[:, 1], m2.predict_proba(X_hold)[:, 1]))

Bản MLflow mới lưu mô hình scikit-learn bằng **skops** thay vì pickle — đúng tinh thần phần 3 — nên ta phải khai báo các kiểu đã duyệt (`skops_trusted_types`). MLflow lưu thêm **signature** (schema đầu vào/đầu ra) và môi trường (`requirements.txt`, `conda.yaml`) cùng mô hình; `mlflow models serve -m models:/churn@champion` dựng ngay một REST API. Ở chương 40 ta tự viết API bằng FastAPI để hiểu từng lớp.

## 9. ONNX: xuất mô hình khỏi Python

ONNX là định dạng **đồ thị tính toán** chuẩn: các phép toán (MatMul, Add, Sigmoid…) + trọng số. Một file ONNX chạy được bằng ONNX Runtime trong C++, C#, Java, JavaScript, trên mobile — không cần Python hay scikit-learn, và **không chứa code** tuỳ ý. Thư viện `skl2onnx` / `torch.onnx.export` tự chuyển đổi; để hiểu bản chất, ta tự dựng đồ thị cho `StandardScaler → LogisticRegression`:

$$p = \sigma\Big(\tfrac{x - \mu}{s}\,W + b\Big)$$

In [ ]:
try:
    import onnx
    import onnxruntime as ort
    from onnx import TensorProto, helper, numpy_helper
except ImportError:
    onnx = None
    print('Chưa cài onnx/onnxruntime — bỏ qua: pip install -e ".[mlops]"')

if onnx is not None:
    numeric_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_tr[NUMERIC].values, y_tr)
    scaler, clf = numeric_model[0], numeric_model[1]
    weights = [numpy_helper.from_array(a.astype(np.float32), name) for name, a in
               [("mean", scaler.mean_), ("scale", scaler.scale_), ("W", clf.coef_.T), ("b", clf.intercept_)]]
    nodes = [helper.make_node("Sub", ["x", "mean"], ["centered"]),
             helper.make_node("Div", ["centered", "scale"], ["z"]),
             helper.make_node("MatMul", ["z", "W"], ["linear"]),
             helper.make_node("Add", ["linear", "b"], ["logit"]),
             helper.make_node("Sigmoid", ["logit"], ["probability"])]
    graph = helper.make_graph(nodes, "churn_logreg",
                              [helper.make_tensor_value_info("x", TensorProto.FLOAT, [None, len(NUMERIC)])],
                              [helper.make_tensor_value_info("probability", TensorProto.FLOAT, [None, 1])], weights)
    onnx_model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)], producer_name="ml-course")
    onnx_model.ir_version = 9
    onnx.checker.check_model(onnx_model)
    onnx_path = WORKDIR / "churn_logreg.onnx"
    onnx.save(onnx_model, onnx_path)

    session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    onnx_proba = session.run(None, {"x": X_te[NUMERIC].values.astype(np.float32)})[0][:, 0]
    sk_proba = numeric_model.predict_proba(X_te[NUMERIC].values)[:, 1]
    print(f"file ONNX: {onnx_path.stat().st_size} byte | sai khác lớn nhất so với scikit-learn: {np.abs(onnx_proba - sk_proba).max():.1e} (float32)")
    print("các phép toán trong đồ thị:", [n.op_type for n in onnx_model.graph.node])

In [ ]:
if onnx is not None:
    import matplotlib.pyplot as plt

    from ml_course.visualization import set_style

    set_style()
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].scatter(sk_proba, onnx_proba, s=6, alpha=0.5)
    axes[0].plot([0, 1], [0, 1], "k--", lw=1)
    axes[0].set_xlabel("xác suất — scikit-learn (float64)")
    axes[0].set_ylabel("xác suất — ONNX Runtime (float32)")
    axes[0].set_title("Hai bản mô hình cho cùng dự đoán")
    axes[1].hist(onnx_proba - sk_proba, bins=40, color="C1")
    axes[1].set_xlabel("ONNX − scikit-learn")
    axes[1].set_title("Sai khác chỉ ở mức làm tròn float32")
    plt.tight_layout()
    plt.show()

Sai khác ~1e-7 là do ONNX tính bằng float32 còn scikit-learn dùng float64 — vì vậy khi kiểm thử mô hình đã chuyển đổi, so sánh **có dung sai** (bài 02).

## 🧠 Kiểm tra nhanh

1. Vì sao phải lưu cả pipeline tiền xử lý cùng mô hình?
2. Vì sao mô hình chứa hàm định nghĩa trong notebook không nạp được ở server? Cách khắc phục?
3. Vì sao `pickle.load` file lạ là nguy hiểm? Ba lựa chọn an toàn hơn?
4. Metadata của một gói mô hình nên chứa gì?
5. Vì sao cần kiểm tra đầu vào khi mô hình "không bao giờ báo lỗi"? Hai lớp kiểm tra?
6. Registry, phiên bản, alias là gì? Rollback hoạt động thế nào?
7. Cổng thăng hạng nên kiểm tra những gì ngoài AUC tổng?

## 📌 Tóm tắt

- Đơn vị triển khai = pipeline dữ liệu thô → dự đoán; code tuỳ biến nằm trong package có phiên bản; kiểm chứng bằng cách nạp ở tiến trình mới.
- Pickle thực thi code khi nạp — chỉ nạp file tin cậy, kiểm tra hash; skops/safetensors/ONNX an toàn hơn.
- Gói mô hình kèm metadata (phiên bản, schema, metric, hash dữ liệu & file) và model card sinh tự động.
- Kiểm tra đầu vào bằng schema nghiệp vụ (pydantic) và schema học từ dữ liệu train.
- Registry: phiên bản bất biến + alias; thăng hạng qua cổng kiểm tra trên holdout cố định (tổng, theo nhóm, độ trễ, schema); rollback = đổi alias.

## 📚 Tài liệu tham khảo

- Mitchell et al. (2019) — *Model Cards for Model Reporting*
- Gebru et al. (2021) — *Datasheets for Datasets*
- Tài liệu Python — `pickle`: cảnh báo bảo mật; tài liệu skops — *Secure persistence*
- scikit-learn User Guide — *Model persistence*
- Tài liệu MLflow — *Model Registry*; ONNX — onnx.ai, ONNX Runtime
- Huyen (2022) — *Designing Machine Learning Systems*, chương 7 (Model Deployment)